In [1]:
from pathlib import Path
script_path = Path.cwd() / 'train_model_v4_fix.py'
print(script_path)

c:\GitHubMain\GraduateWork\ml4\train_model_v4_fix.py


In [2]:
namespace = {}
exec(script_path.read_text(encoding='utf-8'), namespace)
build_feature_table = namespace['build_feature_table']
train_model = namespace['train_model']

In [3]:
datasets_root = Path.cwd() / 'datasets'
print('Datasets root:', datasets_root)
df, audit = build_feature_table(datasets_root)
print('Rows:', len(df))
print('Unique datasets:', df['dataset_id'].nunique())
print('Новые признаки в датасете:', [c for c in df.columns if 'relative' in c or 'ratio' in c or 'log' in c or 'n_children' in c])
df.head()

Datasets root: c:\GitHubMain\GraduateWork\ml4\datasets
Rows: 1233
Unique datasets: 29
Новые признаки в датасете: []


,dataset_id,node_id,target_bootstrap,branch_length,depth,n_leaves_subtree,subtree_fraction,subtree_balance,mean_child_branch_length,std_child_branch_length,taxa_count,alignment_length,gap_fraction_global,gc_mean_global,gc_std_global,variable_site_fraction_global,gap_fraction_clade,gc_mean_clade,gc_std_clade,mean_pairwise_pdist_clade
0,dt002,dt002_node_0,35.2,0.011870,0.011870,11.0,5.5,0.272727,0.014386,0.009330,2.0,16.0,0.25,0.3125,0.1875,0.75,NaN,NaN,NaN,NaN
1,dt002,dt002_node_1,24.6,0.005056,0.016926,7.0,3.5,0.142857,0.029377,0.017885,2.0,16.0,0.25,0.3125,0.1875,0.75,NaN,NaN,NaN,NaN
2,dt002,dt002_node_2,70.8,0.023715,0.035585,4.0,2.0,0.500000,0.000000,0.000000,2.0,16.0,0.25,0.3125,0.1875,0.75,NaN,NaN,NaN,NaN
3,dt002,dt002_node_3,47.3,0.011492,0.028418,4.0,2.0,0.500000,0.054425,0.031047,2.0,16.0,0.25,0.3125,0.1875,0.75,NaN,NaN,NaN,NaN
4,dt002,dt002_node_4,99.4,0.047263,0.064188,3.0,1.5,0.333333,0.000933,0.000933,2.0,16.0,0.25,0.3125,0.1875,0.75,NaN,NaN,NaN,NaN


In [4]:
import pandas as pd
pd.DataFrame(audit)

,dataset_id,alignment_file,tree_file,ok,error,n_rows
0,dt002,c:\GitHubMain\GraduateWork\ml4\datasets\dt002\...,c:\GitHubMain\GraduateWork\ml4\datasets\dt002\...,True,None,10.0
1,dt003,c:\GitHubMain\GraduateWork\ml4\datasets\dt003\...,c:\GitHubMain\GraduateWork\ml4\datasets\dt003\...,True,None,34.0
2,dt004,c:\GitHubMain\GraduateWork\ml4\datasets\dt004\...,c:\GitHubMain\GraduateWork\ml4\datasets\dt004\...,True,None,25.0
3,dt007,c:\GitHubMain\GraduateWork\ml4\datasets\dt007\...,c:\GitHubMain\GraduateWork\ml4\datasets\dt007\...,True,None,110.0
4,dt010,c:\GitHubMain\GraduateWork\ml4\datasets\dt010\...,c:\GitHubMain\GraduateWork\ml4\datasets\dt010\...,True,None,14.0
5,dt012,c:\GitHubMain\GraduateWork\ml4\datasets\dt012\...,c:\GitHubMain\GraduateWork\ml4\datasets\dt012\...,True,None,28.0
6,dt017,c:\GitHubMain\GraduateWork\ml4\datasets\dt017\...,c:\GitHubMain\GraduateWork\ml4\datasets\dt017\...,True,None,62.0
7,dt020,c:\GitHubMain\GraduateWork\ml4\datasets\dt020\...,c:\GitHubMain\GraduateWork\ml4\datasets\dt020\...,True,None,24.0
8,dt022,c:\GitHubMain\GraduateWork\ml4\datasets\dt022\...,c:\GitHubMain\GraduateWork\ml4\datasets\dt022\...,True,None,15.0
9,dt024,c:\GitHubMain\GraduateWork\ml4\datasets\dt024\...,c:\GitHubMain\GraduateWork\ml4\datasets\dt024\...,True,None,43.0


In [5]:
model, metrics, feature_importance, pred_df = train_model(df)

metrics

feature_importance.head(20)

[INFO] Запуск подбора гиперпараметров для Random Forest...
Fitting 5 folds for each of 50 candidates, totalling 250 fits


,feature,importance
0,branch_length,0.556782
5,mean_child_branch_length,0.087010
2,n_leaves_subtree,0.042381
6,std_child_branch_length,0.041515
21,deep_node,0.032270
1,depth,0.031680
3,subtree_fraction,0.024910
9,gap_fraction_global,0.022403
15,gc_std_clade,0.022054
16,mean_pairwise_pdist_clade,0.020158


In [6]:
import json
import joblib
import pandas as pd
from pathlib import Path

output_dir = Path.cwd() / "ml_outputs_v1"
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / "node_dataset.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(audit).to_csv(output_dir / "dataset_audit.csv", index=False, encoding="utf-8-sig")
feature_importance.to_csv(output_dir / "feature_importance.csv", index=False, encoding="utf-8-sig")
pred_df.to_csv(output_dir / "test_predictions.csv", index=False, encoding="utf-8-sig")
joblib.dump(model, output_dir / "model_v1.pkl")

with open(output_dir / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Сохранено в:", output_dir)

Сохранено в: c:\GitHubMain\GraduateWork\ml4\ml_outputs_v1


In [7]:
print("Всего строк:", len(df))
print("Уникальных datasets:", df["dataset_id"].nunique())
print("Колонки:")
print(df.columns.tolist())

Всего строк: 1233
Уникальных datasets: 29
Колонки:
['dataset_id', 'node_id', 'target_bootstrap', 'branch_length', 'depth', 'n_leaves_subtree', 'subtree_fraction', 'subtree_balance', 'mean_child_branch_length', 'std_child_branch_length', 'taxa_count', 'alignment_length', 'gap_fraction_global', 'gc_mean_global', 'gc_std_global', 'variable_site_fraction_global', 'gap_fraction_clade', 'gc_mean_clade', 'gc_std_clade', 'mean_pairwise_pdist_clade']


In [8]:
df["target_bootstrap"].describe()

count    1233.000000
mean       61.220024
std        32.269307
min         0.000000
25%        35.000000
50%        62.670000
75%        96.330000
max       100.000000
Name: target_bootstrap, dtype: float64

In [9]:
metrics

{'n_rows_total': 1233,
 'mae': 7.948083539724029,
 'rmse': 10.911701215877864,
 'r2': 0.8852299872166431,
 'cv_r2_mean': 0.4834519112015636,
 'best_params': {'max_depth': 12,
  'max_features': 0.8,
  'min_samples_leaf': 5,
  'min_samples_split': 11,
  'n_estimators': 420},
 'pred_range': '11.3–99.9',
 'feature_columns': ['branch_length',
  'depth',
  'n_leaves_subtree',
  'subtree_fraction',
  'subtree_balance',
  'mean_child_branch_length',
  'std_child_branch_length',
  'taxa_count',
  'alignment_length',
  'gap_fraction_global',
  'gc_mean_global',
  'gc_std_global',
  'variable_site_fraction_global',
  'gap_fraction_clade',
  'gc_mean_clade',
  'gc_std_clade',
  'mean_pairwise_pdist_clade',
  'bl_cv',
  'gap_var_clade',
  'gc_extreme',
  'tiny_clade',
  'deep_node']}